<a href="https://colab.research.google.com/github/SarahkhIT/AgentsEngineeringProject/blob/main/notebooks/03_production_persistence_and_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Production Readiness: Persistence & Cloud Deployment
**Solar Farm Agentic System: Part 3 of 5**

Covers **Rubric Deliverable 5 (Production Readiness: Persistence, HITL & Cloud)**.
(HITL itself — the `interrupt`/resume — is demonstrated in `02_graph_orchestration_and_hitl.ipynb`;
this notebook covers the report-persistence and cloud-deployment side.)

> **Run order:** continues from `02_graph_orchestration_and_hitl.ipynb` — the
> `save_report()` test below uses the `result` dict produced by that
> notebook's graph run. Run `01` → `02` → `03` in order in the same kernel.
> If you're running this notebook standalone, the fallback cell right below
> creates a minimal sample `result` so the rest still runs.

Sets up a real SQLite database for analysis reports (separate from the
LangGraph checkpointer), a human-approval update function, a FastAPI backend
exposing `/`, `/reports`, and `/reports/{id}/approval`, and the deployment
artifacts that prove a real cloud story: `requirements.txt`, `Dockerfile`,
`docker-compose.yml`, and a standalone `api.py`.


In [ ]:
# Backend & Persistence
# SQLite Database

import sqlite3
from datetime import datetime

DB_NAME = "solar_farm.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS analysis_reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        task TEXT,
        weather_condition TEXT,
        irradiance REAL,
        panel_group TEXT,
        output_pct REAL,
        fault INTEGER,
        predicted_kwh REAL,
        maintenance_needed INTEGER,
        approval_status TEXT,
        created_at TEXT
    )
    """)

    conn.commit()
    conn.close()

init_db()

print("SQLite database created successfully!")

SQLite database created successfully!


In [ ]:
# Save Analysis Report

def save_report(result):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    weather = result["weather_data"]
    panel = result["panel_status"]["group_12"]
    forecast = result["energy_forecast"]

    cursor.execute("""
    INSERT INTO analysis_reports (
        task,
        weather_condition,
        irradiance,
        panel_group,
        output_pct,
        fault,
        predicted_kwh,
        maintenance_needed,
        approval_status,
        created_at
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        result["task"],
        weather["condition"],
        weather["irradiance"],
        "group_12",
        panel["output_pct"],
        int(panel["fault"]),
        forecast["predicted_kwh"],
        int(result["maintenance_needed"]),
        "Pending",
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ))

    conn.commit()
    conn.close()

    print("Report saved successfully!")

In [ ]:
# Fallback: only runs if this notebook is opened standalone, without
# 02_graph_orchestration_and_hitl.ipynb's `result` already in memory.
if "result" not in globals():
    result = {
        "task": "Assess solar farm status",
        "weather_data": {"source": "open-meteo", "condition": "cloudy",
                          "irradiance": 0.0, "cloud_cover_pct": 88, "temperature_c": 42.1},
        "panel_status": {"group_12": {"expected_kw": 10.0, "actual_kw": 8.2,
                                       "output_pct": 82.0, "fault": True}},
        "energy_forecast": {"predicted_kwh": 0.0, "confidence": 0.83},
        "maintenance_needed": True,
    }
    print("Standalone run detected — using a sample `result` dict.")


In [ ]:
# Test Saving Report

save_report(result)

Report saved successfully!


In [ ]:
# Verify Saved Reports

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("SELECT * FROM analysis_reports")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

(1, 'Assess solar farm status', 'cloudy', 0.0, 'group_12', 82.0, 1, 0.0, 1, 'Rejected', '2026-08-05 18:11:09')
(2, 'Assess solar farm status', 'cloudy', 0.0, 'group_12', 82.0, 1, 0.0, 1, 'Pending', '2026-08-05 18:11:41')


In [ ]:
# Human-in-the-Loop Approval

def update_approval(report_id: int, status: str):
    allowed_statuses = ["Approved", "Rejected"]

    if status not in allowed_statuses:
        raise ValueError("Status must be Approved or Rejected")

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    UPDATE analysis_reports
    SET approval_status = ?
    WHERE id = ?
    """, (status, report_id))

    conn.commit()
    conn.close()

    print(f"Approval status updated to: {status}")

In [ ]:
# Test Human-in-the-Loop Approval

report_id = 1
approval = "Approved"

update_approval(report_id, approval)

Approval status updated to: Approved


In [ ]:
# Verify Approval Status

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("""
SELECT id, approval_status
FROM analysis_reports
WHERE id = 1
""")

print(cursor.fetchone())

conn.close()

(1, 'Approved')


In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok

In [ ]:
# FastAPI Backend

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

api = FastAPI(
    title="Solar Farm AI API",
    version="1.0.0"
)

class AnalysisRequest(BaseModel):
    task: str = "Assess solar farm status"

class ApprovalRequest(BaseModel):
    status: str

In [ ]:
# API Endpoints

@api.get("/")
def root():
    return {
        "message": "Solar Farm AI API is running"
    }


@api.get("/reports")
def get_reports():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute("""
    SELECT *
    FROM analysis_reports
    ORDER BY id DESC
    """)

    rows = cursor.fetchall()
    conn.close()

    return [dict(row) for row in rows]


@api.get("/reports/{report_id}")
def get_report(report_id: int):
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute("""
    SELECT *
    FROM analysis_reports
    WHERE id = ?
    """, (report_id,))

    row = cursor.fetchone()
    conn.close()

    if row is None:
        raise HTTPException(
            status_code=404,
            detail="Report not found"
        )

    return dict(row)


@api.post("/reports/{report_id}/approval")
def approve_report(report_id: int, request: ApprovalRequest):
    status = request.status.capitalize()

    if status not in ["Approved", "Rejected"]:
        raise HTTPException(
            status_code=400,
            detail="Status must be Approved or Rejected"
        )

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id
    FROM analysis_reports
    WHERE id = ?
    """, (report_id,))

    if cursor.fetchone() is None:
        conn.close()
        raise HTTPException(
            status_code=404,
            detail="Report not found"
        )

    cursor.execute("""
    UPDATE analysis_reports
    SET approval_status = ?
    WHERE id = ?
    """, (status, report_id))

    conn.commit()
    conn.close()

    return {
        "report_id": report_id,
        "approval_status": status
    }

In [ ]:
# API Testing

from fastapi.testclient import TestClient

client = TestClient(api)

In [ ]:
# Test Root Endpoint

response = client.get("/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Solar Farm AI API is running'}


In [ ]:
# Test Reports Endpoint

response = client.get("/reports")

print("Status code:", response.status_code)
print("Reports:", response.json())

Status code: 200
Reports: [{'id': 2, 'task': 'Assess solar farm status', 'weather_condition': 'cloudy', 'irradiance': 0.0, 'panel_group': 'group_12', 'output_pct': 82.0, 'fault': 1, 'predicted_kwh': 0.0, 'maintenance_needed': 1, 'approval_status': 'Pending', 'created_at': '2026-08-05 18:11:41'}, {'id': 1, 'task': 'Assess solar farm status', 'weather_condition': 'cloudy', 'irradiance': 0.0, 'panel_group': 'group_12', 'output_pct': 82.0, 'fault': 1, 'predicted_kwh': 0.0, 'maintenance_needed': 1, 'approval_status': 'Approved', 'created_at': '2026-08-05 18:11:09'}]


In [ ]:
# Test Approval Endpoint

response = client.post(
    "/reports/1/approval",
    json={"status": "Rejected"}
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'report_id': 1, 'approval_status': 'Rejected'}


In [ ]:
# Create requirements.txt for Docker deployment

requirements_content = """
fastapi
uvicorn[standard]
pydantic
langgraph
langchain
langchain-openai
"""

with open("requirements.txt", "w") as file:
    file.write(requirements_content.strip())

print("requirements.txt created successfully")

requirements.txt created successfully


In [ ]:
# Create Dockerfile

dockerfile_content = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "api:api", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w") as file:
    file.write(dockerfile_content.strip())

print("Dockerfile created successfully")

Dockerfile created successfully


In [ ]:
# Create docker-compose.yml

compose_content = """
services:
  solar-farm-api:
    build: .
    container_name: solar-farm-api
    ports:
      - "8000:8000"
    volumes:
      - ./solar_farm.db:/app/solar_farm.db
    restart: unless-stopped
"""

with open("docker-compose.yml", "w") as file:
    file.write(compose_content.strip())

print("docker-compose.yml created successfully")

docker-compose.yml created successfully


In [ ]:
# Verify Docker deployment files

import os

docker_files = [
    "requirements.txt",
    "Dockerfile",
    "docker-compose.yml"
]

for filename in docker_files:
    print(f"{filename}: {os.path.exists(filename)}")

requirements.txt: True
Dockerfile: True
docker-compose.yml: True


In [ ]:
# Display Dockerfile

with open("Dockerfile", "r") as file:
    print(file.read())

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "api:api", "--host", "0.0.0.0", "--port", "8000"]


In [ ]:
# Display docker-compose.yml

with open("docker-compose.yml", "r") as file:
    print(file.read())

services:
  solar-farm-api:
    build: .
    container_name: solar-farm-api
    ports:
      - "8000:8000"
    volumes:
      - ./solar_farm.db:/app/solar_farm.db
    restart: unless-stopped


In [ ]:
api = FastAPI()

In [ ]:
with open("api.py", "r") as f:
    print(f.read())

import sqlite3

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


DB_NAME = "solar_farm.db"


def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS analysis_reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        task TEXT,
        weather_condition TEXT,
        irradiance REAL,
        panel_group TEXT,
        output_pct REAL,
        fault INTEGER,
        predicted_kwh REAL,
        maintenance_needed INTEGER,
        approval_status TEXT,
        created_at TEXT
    )
    """)

    conn.commit()
    conn.close()


init_db()


api = FastAPI(
    title="Solar Farm AI API",
    version="1.0.0"
)


class AnalysisRequest(BaseModel):
    task: str = "Assess solar farm status"


class ApprovalRequest(BaseModel):
    status: str


@api.get("/")
def root():
    return {
        "message": "Solar Farm AI API is running"
    }


@api.get("/reports")
def get_reports():
    conn = 

In [ ]:
# Create api.py for Docker deployment

api_code = '''
import sqlite3

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


DB_NAME = "solar_farm.db"


def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS analysis_reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        task TEXT,
        weather_condition TEXT,
        irradiance REAL,
        panel_group TEXT,
        output_pct REAL,
        fault INTEGER,
        predicted_kwh REAL,
        maintenance_needed INTEGER,
        approval_status TEXT,
        created_at TEXT
    )
    """)

    conn.commit()
    conn.close()


init_db()


api = FastAPI(
    title="Solar Farm AI API",
    version="1.0.0"
)


class AnalysisRequest(BaseModel):
    task: str = "Assess solar farm status"


class ApprovalRequest(BaseModel):
    status: str


@api.get("/")
def root():
    return {
        "message": "Solar Farm AI API is running"
    }


@api.get("/reports")
def get_reports():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute("""
    SELECT *
    FROM analysis_reports
    ORDER BY id DESC
    """)

    rows = cursor.fetchall()
    conn.close()

    return [dict(row) for row in rows]


@api.get("/reports/{report_id}")
def get_report(report_id: int):
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    cursor.execute("""
    SELECT *
    FROM analysis_reports
    WHERE id = ?
    """, (report_id,))

    row = cursor.fetchone()
    conn.close()

    if row is None:
        raise HTTPException(
            status_code=404,
            detail="Report not found"
        )

    return dict(row)


@api.post("/reports/{report_id}/approval")
def approve_report(report_id: int, request: ApprovalRequest):
    status = request.status.capitalize()

    if status not in ["Approved", "Rejected"]:
        raise HTTPException(
            status_code=400,
            detail="Status must be Approved or Rejected"
        )

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id
    FROM analysis_reports
    WHERE id = ?
    """, (report_id,))

    if cursor.fetchone() is None:
        conn.close()
        raise HTTPException(
            status_code=404,
            detail="Report not found"
        )

    cursor.execute("""
    UPDATE analysis_reports
    SET approval_status = ?
    WHERE id = ?
    """, (status, report_id))

    conn.commit()
    conn.close()

    return {
        "report_id": report_id,
        "approval_status": status
    }
'''

with open("api.py", "w") as file:
    file.write(api_code.strip())

print("api.py created successfully")

api.py created successfully


In [ ]:
# Verify api.py

import py_compile

py_compile.compile("api.py", doraise=True)

print("api.py is valid")

api.py is valid
